# 문장 유사도를 이용한 chatbot 개발

In [ ]:
# 학습: gpu 연결

## 데이터 불러오기

In [ ]:
# colab과 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd

# 파일 경로 설정
file_path='/content/drive/MyDrive/sk/ChatBotData.csv'

# DataFrame 생성
df = pd.read_csv(file_path)
display(df)

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0
...,...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!,2
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.,2
11820,흑기사 해주는 짝남.,설렜겠어요.,2
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.,2


In [ ]:
# 불필요한 컬럼 삭제
df1 = df.drop(columns=['label'])
display(df1)

,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.
...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.
11820,흑기사 해주는 짝남.,설렜겠어요.
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.


## SentenceTransformer: 문장 임베딩 백터 생성

In [ ]:
# 필요한 모델 클래스 임포트
from sentence_transformers import SentenceTransformer

In [ ]:
# 사전 학습된 한국어 임베딩 모델 생성
model_name='jhgan/ko-sroberta-multitask'
model=SentenceTransformer(model_name)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# 예상 질문(Q)에 대한 임베딩 백터 생성

## 필요한 함수 임포트 (진행율 표시)
from tqdm import tqdm

## Q 컬럼 인덱싱 -> 리스트 자료형 변환
sentences = df1['Q'].tolist()
print(sentences)

print('-'*80)

## 생성된 임베딩 백터 저장 -> 빈 리스트 생성
embedding_list=[]

## for문 + append() 사용
for sentence in tqdm(sentences):
    # 예상 질문 문장 1개 -> 임베딩 백터(1차원 배열) 생성
    embedding = model.encode(sentence)
    # 임베딩 백터 저장
    embedding_list.append(embedding)

## 결과 확인
# print(f'생성된 임베딩 백터: \n{embedding_list}')
# print('-'*80)
print(f'생성된 임베딩 백터의 개수: {len(embedding_list)}개')

## 새로운 컬럼 생성

In [ ]:
# df -> 새로운 컬럼 -> df['embedding'] = 값(예상 질문 Q에 대한 임베딩 백터의 리스트)
df1['embedding'] = embedding_list
display(df1)

,Q,A,embedding
0,12시 땡!,하루가 또 가네요.,"[-0.19072695, 0.3214757, 0.59873754, 0.4461786..."
1,1지망 학교 떨어졌어,위로해 드립니다.,"[0.12320574, 0.6423682, 0.6853603, -0.42912346..."
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,"[0.024357503, -0.22393543, 0.22917706, -0.4501..."
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,"[0.036285225, -0.21048254, 0.07269403, -0.5403..."
4,PPL 심하네,눈살이 찌푸려지죠.,"[0.022030605, 0.7055219, -0.17157203, 0.165971..."
...,...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!,"[0.16655469, -0.1795471, 0.24957223, -0.467023..."
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.,"[0.16655469, -0.1795471, 0.24957223, -0.467023..."
11820,흑기사 해주는 짝남.,설렜겠어요.,"[-0.35614997, -0.11077462, 0.33425432, 0.97426..."
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.,"[-0.8464917, -0.4659174, -0.0419846, 0.0322666..."


## 사용자 정의 함수 생성

In [8]:
# 1차원 배열 간의 코사인 유사도 측정 함수

import numpy as np

def cosine_similarity_np(v1, v2):
    v1 = np.asarray(v1)
    v2 = np.asarray(v2)

    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)

    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0

    return dot_product / (norm_v1 * norm_v2)

1.0


In [13]:
# 사용자 질문에 대한 응답 구현 함수 작동 테스트

## 사용자 질문(예시) 생성
question = "배가 고파요"

## 사용자 질문 -> 임베딩 백터 생성
user_embedding = model.encode(question)
# print(f'사용자 질문에 대한 임베딩의 결과: \n{user_embedding}')

## 코사인유사도 계산: 사용자 질문과 예상 질문의 임베딩 백터간의 유사도
similarity_list=[]
for embedding in df1['embedding']:
    # 사용자 질문과 예상 질문 간의 코사인 유사도 계산
    sim = cosine_similarity_np(user_embedding, embedding)
    # 결과(유사도)를 simialrity_list에 저장
    similarity_list.append(sim)

print(f'사용자 질문과 예상 질문간의 코사인 유사도: \n{similarity_list}')

## similarity 컬럼 생성
df1['similarity'] = similarity_list
display(df1)

print('-'*80)

## similarity 컬럼 -> 값이 최대인 행 인덱스 추출 -> idxmax() 함수 사용
idx = df1['similarity'].idxmax()
print(f'사용자 질문과 가장 유사한 예상 질문의 행 인덱스: {idx}')

print('-'*80)

## 코사인 유사도 값이 가장 큰 예상 질문에 대한 답변 추출 -> 행: idx, 열: A -> 인덱싱
answer = df1.loc[idx, 'A']
print(f'챗봇의 답변: {answer}')

사용자 질문과 예상 질문간의 코사인 유사도: 
[np.float32(0.25410923), np.float32(0.1034284), np.float32(0.19564804), np.float32(0.14702974), np.float32(0.2556252), np.float32(0.033377424), np.float32(0.047096264), np.float32(0.061379265), np.float32(0.22025976), np.float32(0.25677532), np.float32(0.221525), np.float32(0.3224621), np.float32(0.25276542), np.float32(0.11219427), np.float32(0.30981368), np.float32(0.36268452), np.float32(0.059412736), np.float32(0.31909168), np.float32(0.32210305), np.float32(0.37031847), np.float32(0.3370993), np.float32(0.30965984), np.float32(0.10106498), np.float32(0.20521551), np.float32(0.24267286), np.float32(0.1662438), np.float32(0.20566559), np.float32(0.17772359), np.float32(0.24112588), np.float32(0.32173416), np.float32(0.24148367), np.float32(0.23666263), np.float32(0.16518822), np.float32(0.22071843), np.float32(0.26723552), np.float32(0.20950812), np.float32(0.24934925), np.float32(0.22185504), np.float32(0.1093562), np.float32(0.24446526), np.float32(0.2977

,Q,A,embedding,similarity
0,12시 땡!,하루가 또 가네요.,"[-0.19072695, 0.3214757, 0.59873754, 0.4461786...",0.254109
1,1지망 학교 떨어졌어,위로해 드립니다.,"[0.12320574, 0.6423682, 0.6853603, -0.42912346...",0.103428
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,"[0.024357503, -0.22393543, 0.22917706, -0.4501...",0.195648
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,"[0.036285225, -0.21048254, 0.07269403, -0.5403...",0.147030
4,PPL 심하네,눈살이 찌푸려지죠.,"[0.022030605, 0.7055219, -0.17157203, 0.165971...",0.255625
...,...,...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!,"[0.16655469, -0.1795471, 0.24957223, -0.467023...",0.180162
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.,"[0.16655469, -0.1795471, 0.24957223, -0.467023...",0.180162
11820,흑기사 해주는 짝남.,설렜겠어요.,"[-0.35614997, -0.11077462, 0.33425432, 0.97426...",0.105279
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.,"[-0.8464917, -0.4659174, -0.0419846, 0.0322666...",0.139800


--------------------------------------------------------------------------------
사용자 질문과 가장 유사한 예상 질문의 행 인덱스: 1998
--------------------------------------------------------------------------------
챗봇의 답변: 뭐 좀 챙겨드세요.


In [14]:
# 사용자 질문에 대한 응답 구현 함수 정의

def chatbot(question):
    # 사용자 질문 -> 임베딩 백터 생성
    user_embedding = model.encode(question)
    # 사용자 질문과 예상 질문('Q')간의 코사인 유사도
    similarity_list=[]
    for embedding in df1['embedding']:
        sim=cosine_similarity_np(user_embedding, embedding)
        similarity_list.append(sim)
    # similarity 컬럼 생성 -> 코사인 유사도 값 저장
    df1['similarity'] = similarity_list
    # similarity 컬럼 -> 값이 최대인 행 인덱스 추출
    idx = df1['similarity'].idxmax()
    # 답변 추출 -> 행: idx, 열: 'A' -> 인덱싱
    answer = df1.loc[idx, 'A']
    # 답변 반환
    return answer

## chatbot 함수 실행

In [17]:
# 사용자 질문
# question = '배가 고파요'
# question = '출근하기 싫어요'
question = '여행가고 싶다'

# 챗봇 응답
answer = chatbot(question)
print(f'chatbot: {answer}')

chatbot: 계획을 세워보세요.
